**Using sentence-transformers for semantic search, which involves embedding documents and queries into a vector space and finding the most similar documents to a given query.**

**Step 1:- Install sentence-transformers**

In [1]:
!pip install sentence-transformers==4.1.0 | tail -n 1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


**Step 2:- Import Libraries**

In [2]:
import math

import numpy as np
import scipy
import torch
from sentence_transformers import SentenceTransformer

**Step 3:- Load Pre-trained Model**

In [3]:
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

**Step 4:- Define Documents**

In [4]:
documents = [
 'Account balance exceeds the approved $50,000 credit limit.',
 'Shipment held at port for harmonized tariff code verification.',
 'Pallet dropped during unloading; three units show structural casing cracks.',
 "Random spot check required for new vendor's first electronics shipment.",
 'Awaiting confirmation of wire transfer for international Order #441.',
 'Missing mandatory safety warning labels for the European market.',
 'Primary SKU is out of stock; pending replenishment from the manufacturer.',
]

**Step 5:- Generate Document Embeddings**

In [5]:
embeddings = model.encode(documents)

**Step 6:- Check Embeddings Shape**

In [6]:
embeddings.shape

(7, 384)

**Step 7:- Calculate L2 Norms**

Calculate the L2 norm (Euclidean norm or magnitude) for each document embedding. The L2 norm is used for normalizing the embeddings.

In [8]:
l2_norms = np.sqrt(np.sum(embeddings**2, axis=1))
l2_norms

array([6.209782 , 6.571253 , 6.0983977, 6.1051884, 6.1901975, 6.597366 ,
       5.615634 ], dtype=float32)

**Step 8:- Reshape L2 Norms**

The L2 norms are reshaped from a 1D array to a 2D array (-1, 1) to enable element-wise division with the embeddings array during normalization.

In [9]:
l2_norms_reshaped = l2_norms.reshape(-1,1)
l2_norms_reshaped

array([[6.209782 ],
       [6.571253 ],
       [6.0983977],
       [6.1051884],
       [6.1901975],
       [6.597366 ],
       [5.615634 ]], dtype=float32)

**Step 9: Manually Normalize Embeddings**

This step manually normalizes the embeddings by dividing each embedding by its corresponding L2 norm. Normalization ensures that all embeddings have a unit length, which is crucial for accurate cosine similarity calculations.

In [10]:
normalized_embeddings_manual = embeddings/l2_norms_reshaped
normalized_embeddings_manual

array([[ 0.03179936,  0.05401345,  0.03085974, ..., -0.03235293,
         0.0281877 , -0.04907949],
       [-0.0479747 ,  0.04871312, -0.07408653, ...,  0.03233325,
         0.01419047, -0.06289357],
       [-0.09334826, -0.02992088,  0.06175756, ...,  0.01544828,
         0.06893307,  0.04141391],
       ...,
       [-0.09324063, -0.04836811, -0.04596137, ..., -0.03854122,
        -0.00838208, -0.03987597],
       [ 0.02361614, -0.0075232 , -0.01407237, ..., -0.07807571,
        -0.03163972,  0.05645761],
       [-0.0398886 , -0.02158343, -0.01331425, ..., -0.14011845,
        -0.04153111,  0.05839787]], dtype=float32)

**Step 10: Query and Find Most Similar Document**

In [11]:
# Encodes a query string into an embedding.
query_embedding = model.encode(
    ["List causes for shipment held"]
)

# Normalize the query embedding:
normalized_query_embedding = torch.nn.functional.normalize(
    torch.from_numpy(query_embedding)
).numpy()

# It calculates the cosine_similarity_q3 between the normalized document embeddings and the normalized query embedding. Cosine similarity measures the angle between two vectors, with a higher value indicating greater similarity.
cosine_similarity_q3 = normalized_embeddings_manual @ normalized_query_embedding.T

# Finds the index of the document with the highest cosine similarity to the query.
highest_cossim_position = cosine_similarity_q3.argmax()

# Retrieves and displays the document most semantically similar to the query.
documents[highest_cossim_position]

'Shipment held at port for harmonized tariff code verification.'